# 03 — Aggregate: Silver → Gold + MSSQL write-back

Builds reporting aggregates in **gold** and writes a summary back to SQL Server.

In [ ]:
import sys
sys.path.insert(0, '/workspace')
from scripts.spark_session import get_spark, JDBC_URL, JDBC_PROPS, SILVER, GOLD
from pyspark.sql import functions as F
spark = get_spark('03_aggregate')

In [ ]:
details = spark.read.parquet(f'{SILVER}/order_details')

revenue = (
    details
    .groupBy('region', 'order_month')
    .agg(
        F.sum('line_total').alias('total_revenue'),
        F.sum('quantity').alias('total_units'),
        F.countDistinct('order_id').alias('order_count'),
    )
    .orderBy('order_month', 'region')
)
revenue.show(truncate=False)

In [ ]:
revenue.write.mode('overwrite').parquet(f'{GOLD}/revenue_by_region')
print(f'✅ Wrote {revenue.count()} gold rows')

In [ ]:
revenue.write.jdbc(url=JDBC_URL, table='dbo.gold_revenue_by_region',
    mode='overwrite', properties=JDBC_PROPS)
print('✅ Wrote gold_revenue_by_region back to MSSQL')

In [ ]:
revenue.createOrReplaceTempView('revenue')
spark.sql("""
    SELECT region, SUM(total_revenue) AS grand_total, SUM(total_units) AS grand_units
    FROM revenue GROUP BY region ORDER BY grand_total DESC
""").show()